In [1]:
# --- INSTALLATION ---
# python -m venv .venv
# .venv\Scripts\activate
# pip install ipykernel numpy pandas sqlalchemy pyodbc seaborn matplotlib

<h1> Setting Up

In [2]:
# --- Import Libraries ---
import numpy
import pandas as pd

import sqlite3
# from sqlalchemy import create_engine

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# --- Load Data ---
db_file = 'customer_churn.db'
server = r'(localdb)\MSSQLLocalDB'
database = 'da_projects'
driver = 'ODBC Driver 18 for SQL Server'

# Connect to Database
# connection_string = rf"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
# engine = create_engine(connection_string, fast_executemany=True)

# Import All Tables
conn = sqlite3.connect(db_file)
query ="""
        SELECT name 
        FROM sqlite_master 
        WHERE type='table';
        """
tables_df = pd.read_sql(sql=query, con=conn)

for table_name in tables_df['name']:
    if table_name.startswith('sqlite_'):
        continue
    
    # Read Table to Dataframe
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn) 
    # df.to_sql(table_name, engine, if_exists='replace', index=False)
    # print(f'Finish Loading Table {table_name} into {database}')
    
    globals()[f'df_{table_name}'] = df
    print(f'Finish Creating Table df_{table_name}')
    
    # Get Tables Columns
    columns = pd.read_sql(f"PRAGMA table_info({table_name})", conn)
    print(columns['name'].tolist(), '\n')
    
conn.close()

Finish Creating Table df_db_customer
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode'] 

Finish Creating Table df_db_subscription
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score'] 

Finish Creating Table df_db_support
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment'] 



In [4]:
display(df_db_customer.info())
display(df_db_customer.head(3))
display(df_db_subscription.info())
display(df_db_subscription.head(3))
display(df_db_support.info())
display(df_db_support.head(3))

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


None

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None


<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     str    
 1   subscription_start_date  21 non-null     str    
 2   subscription_type        21 non-null     str    
 3   renewal_date             21 non-null     str    
 4   plan_type                21 non-null     str    
 5   contract_type            21 non-null     str    
 6   cancellation_date        6 non-null      str    
 7   cancellation_reason      6 non-null      str    
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 1.9 KB


None

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34


<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      str   
 1   complaint_date  9 non-null      str   
 2   escalations     9 non-null      str   
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      str   
dtypes: int64(1), object(1), str(4)
memory usage: 564.0+ bytes


None

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,NaN


<h1> Data Cleaning

<h2> Rename Columns

In [5]:
# df_db_customer = df_db_customer.rename(columns={'name':'customer_name'})

<h2> Drop Columns

In [6]:
# df_db_customer: drop column interests and pincode
df_db_customer = df_db_customer.drop(columns=['interests', 'pincode'])

# df_db_support: drop column col_1, comment
df_db_support = df_db_support.drop(columns=['col_1', 'comment'])

<h2> Change Data Type

In [7]:
# df_db_customer: change column dob to datetime
df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

# df_db_subscription: change column subscription_start_date, renewal_date, cancellation_date to date time
for col in ['subscription_start_date', 'renewal_date', 'cancellation_date']:
    df_db_subscription[col] = pd.to_datetime(df_db_subscription[col])
 
# df_db_support: change column complaint_date to datetime
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])

<h2> Data Standardization

In [8]:
# df_db_customer: column gender
df_db_customer['gender'] = df_db_customer['gender'].replace({'Men' : 'Male', 'Women' : 'Female'})
df_db_customer['gender'].unique()

<StringArray>
['Male', 'Female']
Length: 2, dtype: str

<h2> Fix Missing Value

In [9]:
# df_db_customer: column country
country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_db_customer['country'] = df_db_customer['country'].fillna(country_mapping)

<h1> Feature Engineering and Data Analysis

<h2> Create New Columns

<h2> Join Dataframes

<h2> Export Data

<h2> Data Analysis

<h1> Visualization

<h2> Visualization using Matplotlib

<h2> Visualization using Seaborn

<h2> Pivot Table